In [ ]:
from tensorflow import keras
import numpy as np
import nltk
from copy import deepcopy
from scipy.spatial import distance

In [11]:
# Small sample corpus. In this demo, the model will learn only from this text.
text = 'Once upon a time, there lived two monarchs. The king sat on a throne. Also, the queen sat on a throne. Together, they both were seated on thrones.'

# Safety cap: limits how many n-grams we keep, so memory stays manageable on larger texts.
NGRAM_LIMIT = 10000

# Split text into tokens (words + punctuation).
tokens = nltk.tokenize.word_tokenize(text)

# Lowercase for normalization so 'The' and 'the' map to the same token.
tokens_lc = [x.lower() for x in tokens]

# Build vocabulary mappings.
# INDEXES: token -> integer id
INDEXES = {x: i for i, x in enumerate(sorted(set(tokens_lc)))}

# REV_INDEXES: integer id -> token (inverse lookup for readability/debugging).
REV_INDEXES = {x: i for i, x in INDEXES.items()}

# Create training pairs as adjacent words: (w_t, w_(t+1)).
bigrams = list(nltk.bigrams(tokens_lc))

# Apply cap in case corpus is very large.
bigrams = bigrams[:min(len(bigrams), NGRAM_LIMIT)]

In [17]:
# Split each bigram into input word (x) and target/context word (y).
x_text = [w1 for w1, w2 in bigrams]
y_text = [w2 for w1, w2 in bigrams]

# Convert tokens to integer ids using the vocabulary map.
x_num = [INDEXES[x] for x in x_text]
y_num = [INDEXES[y] for y in y_text]

# One-hot encode ids so each word becomes a vocab-sized indicator vector.
# X: input words, y: target next words.
X = keras.utils.to_categorical(x_num, num_classes=len(INDEXES))
y = keras.utils.to_categorical(y_num, num_classes=len(INDEXES))

In [18]:
# Build a simple feed-forward network for next-word prediction.
m = keras.models.Sequential()

# Input is one-hot (size = vocab). Hidden layers learn dense representations.
m.add(keras.layers.Dense(64, activation='relu', input_dim=len(INDEXES)))
m.add(keras.layers.Dense(32, activation='relu'))

# Output is a probability distribution over the vocabulary.
m.add(keras.layers.Dense(len(INDEXES), activation='softmax'))

# Categorical crossentropy is standard for one-hot multi-class prediction.
m.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Show architecture and parameter counts.
m.summary()

# Train on the bigram dataset.
m.fit(X, y, epochs=100)

/Users/siyuliang/anaconda3/envs/mfa/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_6 (Dense)                 │ (None, 64)             │         1,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 23)             │           759 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,375 (17.09 KB)

 Trainable params: 4,375 (17.09 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 0.0294 - loss: 3.1586 
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.0294 - loss: 3.1455
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0588 - loss: 3.1358
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.0588 - loss: 3.1267
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.0588 - loss: 3.1183
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.0588 - loss: 3.1108
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - accuracy: 0.0588 - loss: 3.1038
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0588 - loss: 3.0970 
Epoch 9/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - accuracy: 0.0588 - loss: 3.0905
Epoch 10/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - accuracy: 0.0882 - loss: 3.0839
Epoch 11/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - accuracy: 0.0882 - loss: 3.0774 
Epoch 12/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - accuracy: 0.0882 - l

In [19]:
# Create a model that outputs internal representations instead of final probabilities.
# Note: clone_model copies architecture only (not trained weights).
newmod = keras.models.clone_model(m)

# Remove the softmax output layer so model output is the last hidden layer vector.
newmod.pop()

<Dense name=dense_8, built=True>

In [20]:
def get_vector(word):
    """Return the model's hidden representation vector for a given word token."""
    # Look up the vocabulary id for this word.
    i = INDEXES[word]

    # Convert id to one-hot, matching the model's expected input shape.
    onehot = keras.utils.to_categorical([i], num_classes=len(INDEXES))

    # Run a forward pass through the truncated model.
    return newmod.predict(onehot)

In [21]:
# Example: inspect the vector representation for 'queen'.
print(get_vector('queen'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 30ms/step
[[0.05421392 0.09613089 0.         0.08370302 0.         0.
  0.         0.02215735 0.10993382 0.         0.         0.06262438
  0.         0.07895324 0.         0.0296639  0.00692888 0.15432943
  0.1837513  0.03642865 0.07898806 0.         0.21796073 0.35216388
  0.0862296  0.         0.23706453 0.16819476 0.         0.
  0.01483726 0.02924669]]


In [22]:
# Example: inspect the vector representation for 'king'.
print(get_vector('king'))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step
[[0.         0.17244141 0.         0.         0.07501049 0.04937252
  0.         0.03087944 0.17160012 0.         0.05590534 0.03174627
  0.         0.         0.02763444 0.         0.02388871 0.02314308
  0.07991572 0.02025889 0.16621831 0.01648681 0.12964101 0.14706522
  0.         0.         0.19059126 0.         0.         0.
  0.         0.13620344]]


In [ ]:
def compute_similarity(word1, word2):
    """
    Compute cosine similarity between two word vectors.
    Returns a value between 0 (no similarity) and 1 (identical).
    """
    # Get vectors for both words and flatten to 1D arrays
    vec1 = get_vector(word1).flatten()
    vec2 = get_vector(word2).flatten()
    
    # Compute cosine distance and convert to similarity
    # cosine_similarity = 1 - cosine_distance
    cosine_dist = distance.cosine(vec1, vec2)
    similarity = 1 - cosine_dist
    
    return similarity

# Test similarity between related words in our corpus
print(f"Similarity between 'king' and 'queen': {compute_similarity('king', 'queen'):.4f}")
print(f"Similarity between 'throne' and 'thrones': {compute_similarity('throne', 'thrones'):.4f}")
print(f"Similarity between 'king' and 'sat': {compute_similarity('king', 'sat'):.4f}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step


ValueError: Input vector should be 1-D.